# Capítulo 4. Regresión lineal simple y múltiple

**Aprendizaje y Clasificación Automática con R**  
**Autor:** Jesús Gilberto Rodríguez Escobedo

Este cuaderno es **independiente y autónomo**: puede abrirse directamente sin ejecutar capítulos anteriores.

1. Ejecute primero la celda **Preparación automática y autónoma del capítulo**.
2. Después ejecute las celdas en orden.
3. Si Colab reinicia la sesión, vuelva a ejecutar desde la primera celda.

[Volver al índice de cuadernos Colab](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/00-indice-colabs.ipynb)


In [ ]:
# Preparación automática y autónoma del capítulo
options(repos = c(CRAN = "https://cloud.r-project.org"))

paquetes_libro <- c(
  "ggplot2", "readr", "dplyr", "tidyr", "stringr", "data.table",
  "class", "rpart", "randomForest", "ranger", "e1071", "naivebayes",
  "neuralnet", "cluster", "caret", "factoextra", "scales", "plotly", "DT"
)
faltantes <- paquetes_libro[!vapply(paquetes_libro, requireNamespace, logical(1), quietly = TRUE)]
if (length(faltantes)) install.packages(faltantes)

dir.create("datos/covid19/procesados", showWarnings = FALSE, recursive = TRUE)
dir.create("datos/covid19/muestras", showWarnings = FALSE, recursive = TRUE)
dir.create("datos/covid19/diccionarios", showWarnings = FALSE, recursive = TRUE)

archivos_colab <- c(
  "util_graficas.R" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/util_graficas.R",
  "datos/atus_ml_preparado.csv" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/atus_ml_preparado.csv",
  "datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz",
  "datos/covid19/muestras/covid19_mexico_2022_muestra.csv.gz" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/covid19/muestras/covid19_mexico_2022_muestra.csv.gz",
  "datos/covid19/diccionarios/diccionario_covid19_ml.csv" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/covid19/diccionarios/diccionario_covid19_ml.csv"
)
for (destino in names(archivos_colab)) {
  if (!file.exists(destino)) download.file(archivos_colab[[destino]], destino, mode = "wb", quiet = TRUE)
}
stopifnot(all(file.exists(names(archivos_colab))))
source("util_graficas.R")
cat("Entorno autónomo listo. R:", R.version.string, "\n")


# Regresión lineal simple y múltiple

La formulación matemática de **regresión lineal simple y múltiple** se desarrolla con mayor profundidad
en los capítulos 9 y 10 de *Fundamentos Matemáticos del Aprendizaje
Automático* [@rodriguez2026fundamentos].

## Objetivos del capítulo

Al terminar este capítulo, el lector podrá:

- distinguir entre regresión y clasificación;
- interpretar una relación lineal entre una variable respuesta y una variable explicativa;
- ajustar una regresión lineal simple en R;
- construir e interpretar un modelo de regresión lineal múltiple;
- evaluar el ajuste mediante $R^2$, $R^2$ ajustado, RMSE y análisis de residuos;
- reconocer cuándo la regresión lineal no es apropiada.

## ¿Por qué estudiar regresión lineal en un libro de clasificación?

La regresión lineal no es un algoritmo de clasificación: su respuesta es numérica y continua. Sin embargo, es una base fundamental del aprendizaje supervisado porque introduce ideas que reaparecen en otros modelos: función de predicción, coeficientes, error, ajuste, generalización y evaluación fuera de muestra.

Además, permite comprender mejor la regresión logística del capítulo siguiente. Ambos modelos construyen una combinación de variables explicativas, aunque difieren en la naturaleza de la respuesta y en la función utilizada para obtener la predicción [@james2021islr].

En regresión lineal se predice una cantidad, por ejemplo el número mensual de accidentes. En clasificación se predice una categoría, por ejemplo si un accidente tuvo víctimas o solo daños.

## Regresión lineal simple

La regresión lineal simple relaciona una variable respuesta $Y$ con una sola variable explicativa $X$:

$$
Y_i = \beta_0 + \beta_1 X_i + \varepsilon_i.
$$

Donde:

- $\beta_0$ es la ordenada al origen;
- $\beta_1$ representa el cambio promedio esperado en $Y$ cuando $X$ aumenta una unidad;
- $\varepsilon_i$ representa la parte no explicada por el modelo.

Los coeficientes se estiman normalmente mediante mínimos cuadrados, buscando minimizar la suma de los errores al cuadrado [@montgomery2021introduction].

## Primer ejemplo didáctico

Supongamos que se desea relacionar el número de horas de estudio con la calificación obtenida.


In [ ]:
horas <- c(2, 3, 4, 5, 6, 7, 8, 9)
calificacion <- c(55, 60, 64, 70, 75, 79, 86, 90)

datos_estudio <- data.frame(
  horas = horas,
  calificacion = calificacion
)

datos_estudio


## Visualizar la relación


In [ ]:
library(ggplot2)
source("util_graficas.R")

ggplot(datos_estudio, aes(x = horas, y = calificacion)) +
  geom_point(size = 3) +
  geom_smooth(method = "lm", se = TRUE) +
  labs(
    title = "Horas de estudio y calificación",
    subtitle = "Ejemplo de una relación aproximadamente lineal",
    x = "Horas de estudio",
    y = "Calificación"
  ) +
  tema_libro()


## Ajustar el modelo simple


In [ ]:
modelo_simple <- lm(
  calificacion ~ horas,
  data = datos_estudio
)

summary(modelo_simple)


## Interpretar los coeficientes


In [ ]:
coeficientes_simple <- coef(modelo_simple)
coeficientes_simple


El modelo estimado tiene la forma:

$$
\widehat{Y} = \widehat{\beta}_0 + \widehat{\beta}_1 X.
$$

La pendiente indica cuántos puntos cambia, en promedio, la calificación por cada hora adicional de estudio. La ordenada al origen es necesaria matemáticamente, aunque no siempre tenga una interpretación práctica dentro del rango observado.

## Realizar una predicción


In [ ]:
nuevo_estudiante <- data.frame(horas = 6.5)

predict(
  modelo_simple,
  newdata = nuevo_estudiante,
  interval = "prediction",
  level = 0.95
)


El intervalo de predicción es más amplio que un intervalo para la media porque considera tanto la incertidumbre del modelo como la variabilidad individual.

## Residuos

El residuo de la observación $i$ es:

$$
e_i = y_i - \widehat{y}_i.
$$


In [ ]:
diagnostico_simple <- data.frame(
  observado = datos_estudio$calificacion,
  predicho = fitted(modelo_simple),
  residuo = residuals(modelo_simple)
)

diagnostico_simple


## Visualizar los residuos


In [ ]:
ggplot(diagnostico_simple, aes(x = predicho, y = residuo)) +
  geom_hline(yintercept = 0, linetype = 2) +
  geom_point(size = 3) +
  labs(
    title = "Residuos frente a valores predichos",
    x = "Valor predicho",
    y = "Residuo"
  ) +
  tema_libro()


Una nube sin patrón claro alrededor de cero es compatible con una relación lineal razonable. Curvaturas, forma de embudo o puntos extremos sugieren revisar el modelo.

## Regresión lineal múltiple

La regresión múltiple incorpora varias variables explicativas:

$$
Y_i = \beta_0 + \beta_1X_{i1} + \beta_2X_{i2} + \cdots + \beta_pX_{ip} + \varepsilon_i.
$$

Cada coeficiente representa el cambio esperado en la respuesta cuando la variable correspondiente aumenta una unidad, **manteniendo constantes las demás variables**.

## Preparar un ejemplo múltiple


In [ ]:
set.seed(2026)

n <- 80

datos_ambientales <- data.frame(
  temperatura = runif(n, 12, 35),
  velocidad_viento = runif(n, 0.5, 8),
  trafico = runif(n, 100, 900)
)

datos_ambientales$pm10 <-
  18 +
  0.9 * datos_ambientales$temperatura -
  2.1 * datos_ambientales$velocidad_viento +
  0.045 * datos_ambientales$trafico +
  rnorm(n, mean = 0, sd = 6)

head(datos_ambientales)


## Ajustar el modelo múltiple


In [ ]:
modelo_multiple <- lm(
  pm10 ~ temperatura + velocidad_viento + trafico,
  data = datos_ambientales
)

summary(modelo_multiple)


## Interpretar los coeficientes múltiples


In [ ]:
tabla_coeficientes <- data.frame(
  termino = names(coef(modelo_multiple)),
  estimacion = as.numeric(coef(modelo_multiple))
)

tabla_coeficientes


Por ejemplo, el coeficiente de `velocidad_viento` estima el cambio promedio de PM10 asociado con un aumento de una unidad en la velocidad del viento, manteniendo constantes la temperatura y el tráfico.

Un coeficiente estadísticamente significativo no demuestra por sí solo una relación causal. La causalidad requiere diseño, conocimiento sustantivo y control adecuado de variables de confusión.

## Separar entrenamiento y prueba

Evaluar el modelo con los mismos datos usados para ajustarlo produce una visión demasiado optimista. Por ello se separan datos de entrenamiento y prueba.


In [ ]:
set.seed(2026)

indices_entrenamiento <- sample(
  seq_len(nrow(datos_ambientales)),
  size = floor(0.8 * nrow(datos_ambientales))
)

entrenamiento_reg <- datos_ambientales[indices_entrenamiento, ]
prueba_reg <- datos_ambientales[-indices_entrenamiento, ]

modelo_multiple_prueba <- lm(
  pm10 ~ temperatura + velocidad_viento + trafico,
  data = entrenamiento_reg
)

prediccion_reg <- predict(
  modelo_multiple_prueba,
  newdata = prueba_reg
)


## Métricas de evaluación

El error cuadrático medio y su raíz se calculan como:

$$
RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i-\widehat{y}_i)^2}.
$$


In [ ]:
rmse <- sqrt(mean((prueba_reg$pm10 - prediccion_reg)^2))
mae <- mean(abs(prueba_reg$pm10 - prediccion_reg))

metricas_regresion <- data.frame(
  metrica = c("RMSE", "MAE"),
  valor = c(rmse, mae)
)

metricas_regresion


## Observado frente a predicho


In [ ]:
comparacion_regresion <- data.frame(
  observado = prueba_reg$pm10,
  predicho = prediccion_reg
)

ggplot(comparacion_regresion, aes(x = observado, y = predicho)) +
  geom_point(size = 3, alpha = 0.8) +
  geom_abline(slope = 1, intercept = 0, linetype = 2) +
  labs(
    title = "Valores observados y predichos",
    subtitle = "La línea diagonal representa predicción perfecta",
    x = "PM10 observado",
    y = "PM10 predicho"
  ) +
  tema_libro()


## Aplicación con datos ATUS agregados

En ATUS cada fila corresponde a un accidente. La base preparada del libro conserva las variables necesarias para los algoritmos de clasificación, pero no incluye entidad ni municipio. Por ello, en este ejemplo los registros se agregan por mes y se modela el número mensual de accidentes registrados.


In [ ]:
library(readr)
library(dplyr)

ruta_atus_ml <- "datos/atus_ml_preparado.csv"

if (!file.exists(ruta_atus_ml)) {
  stop(
    paste(
      "No se encontró datos/atus_ml_preparado.csv.",
      "Ejecute primero la celda de preparación automática."
    )
  )
}

atus_reg <- read_csv(
  ruta_atus_ml,
  show_col_types = FALSE
)

names(atus_reg)


## Construir una base mensual


In [ ]:
variables_necesarias <- c(
  "MES", "ID_HORA", "DIASEMANA", "TIPACCID",
  "accidente_con_victimas"
)

faltantes <- setdiff(variables_necesarias, names(atus_reg))

if (length(faltantes) > 0) {
  stop(
    paste(
      "Faltan variables para el ejemplo:",
      paste(faltantes, collapse = ", ")
    )
  )
}

atus_mensual <- atus_reg |>
  mutate(
    ID_HORA = as.numeric(ID_HORA),
    MES = as.numeric(MES),
    fin_semana = DIASEMANA %in% c(
      "sábado", "sabado", "domingo",
      "Sábado", "Sabado", "Domingo"
    ),
    con_victimas = accidente_con_victimas == "Con víctimas"
  ) |>
  group_by(MES) |>
  summarise(
    accidentes = n(),
    tipos_accidente = n_distinct(TIPACCID, na.rm = TRUE),
    hora_promedio = mean(ID_HORA, na.rm = TRUE),
    proporcion_fin_semana = mean(fin_semana, na.rm = TRUE),
    proporcion_con_victimas = mean(con_victimas, na.rm = TRUE),
    .groups = "drop"
  ) |>
  filter(
    is.finite(accidentes),
    is.finite(tipos_accidente),
    is.finite(hora_promedio),
    is.finite(proporcion_fin_semana),
    is.finite(proporcion_con_victimas)
  )

atus_mensual


## Ajustar la regresión múltiple con ATUS


In [ ]:
modelo_atus_reg <- lm(
  accidentes ~ MES + tipos_accidente +
    hora_promedio + proporcion_fin_semana,
  data = atus_mensual
)

summary(modelo_atus_reg)


El número de accidentes es una variable de conteo. La regresión lineal se utiliza aquí con fines didácticos; en un estudio formal podrían ser más apropiados modelos de Poisson o binomial negativa. Esta comparación ayuda a comprender que la elección del algoritmo depende de la naturaleza de la respuesta.

## Supuestos principales

La interpretación inferencial tradicional de la regresión lineal se apoya en varios supuestos:

1. relación aproximadamente lineal;
2. errores independientes;
3. varianza aproximadamente constante;
4. ausencia de colinealidad extrema;
5. normalidad aproximada de los errores cuando se construyen pruebas e intervalos.

Los supuestos deben analizarse mediante gráficas, conocimiento del problema y medidas diagnósticas; no deben tratarse como una lista mecánica.

## Diferencias entre regresión lineal y logística

| Característica | Regresión lineal | Regresión logística |
|---|---|---|
| Respuesta | Numérica continua | Categórica binaria |
| Salida directa | Cualquier número real | Probabilidad entre 0 y 1 |
| Función usual | Identidad | Logística |
| Evaluación | RMSE, MAE, $R^2$ | Matriz de confusión, sensibilidad, especificidad, AUC |
| Ejemplo | Predecir PM10 | Clasificar accidente con víctimas |

## Laboratorio interactivo: regresión lineal

Modifica el intercepto, la pendiente y el nivel de ruido para observar cómo cambia la nube de puntos y la recta de regresión.


**Laboratorio interactivo:** este bloque se ejecuta en la versión web mediante Shinylive; aquí se conserva el desarrollo reproducible del capítulo.


### Laboratorio disponible en la versión web

El laboratorio permite modificar intercepto, pendiente, ruido, tamaño de muestra y semilla, y comparar la recta verdadera con la estimada.

## Actividades para el lector

1. Modifique el ejemplo simple y prediga la calificación para 7.5 horas de estudio.
2. Retire una variable del modelo ambiental y compare el $R^2$ ajustado y el RMSE.
3. Añada una interacción entre temperatura y velocidad del viento.
4. Analice los residuos del modelo múltiple.
5. Explique por qué no sería correcto utilizar regresión lineal ordinaria para predecir directamente una categoría como `Con víctimas` o `Solo daños`.
6. Compare conceptualmente el modelo ATUS de este capítulo con la regresión logística del capítulo siguiente.

## Conclusiones

La regresión lineal simple explica una respuesta cuantitativa mediante una variable y la regresión múltiple permite considerar varios factores simultáneamente. Su valor en este libro no se limita a la predicción numérica: proporciona el lenguaje básico para comprender coeficientes, errores, validación y generalización, conceptos que continuarán apareciendo en los algoritmos de clasificación.

## Materiales complementarios del capítulo

<!-- colab-capitulo -->

Este capítulo cuenta con un **cuaderno autónomo de Google Colab**. Puede abrirse y ejecutarse de manera independiente, sin necesidad de ejecutar los capítulos anteriores.

[**Abrir este capítulo en Google Colab**](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/04-regresion-lineal.ipynb)

::: <!-- /colab-capitulo -->

| Recurso | Utilidad | Abrir o reproducir | Descargar |
|---|---|---|---|
| Video explicativo | Explicación audiovisual de la regresión lineal simple y múltiple. | [Ver en YouTube](https://www.youtube.com/watch?v=SE6DCeU9h1w) | — |
| Presentación en PDF | Diapositivas para lectura, estudio o exposición. | [Ver PDF](recursos/capitulo-04/capitulo-04-regresion-lineal-simple-multiple.pdf) | [Descargar PDF](recursos/capitulo-04/capitulo-04-regresion-lineal-simple-multiple.pdf){download="capitulo-04-regresion-lineal-simple-multiple.pdf"} |
| Presentación editable | Archivo PowerPoint para utilizarlo en clase o adaptarlo. | [Abrir PPTX](recursos/capitulo-04/capitulo-04-regresion-lineal-simple-multiple.pptx) | [Descargar PPTX](recursos/capitulo-04/capitulo-04-regresion-lineal-simple-multiple.pptx){download="capitulo-04-regresion-lineal-simple-multiple.pptx"} |
| Infografía | Síntesis visual de conceptos, fórmulas e interpretación. | [Ver infografía](recursos/capitulo-04/capitulo-04-regresion-lineal-simple-multiple-infografia.png) | [Descargar PNG](recursos/capitulo-04/capitulo-04-regresion-lineal-simple-multiple-infografia.png){download="capitulo-04-regresion-lineal-simple-multiple-infografia.png"} |

### Video explicativo

### Vista previa de la infografía

[![Infografía del capítulo 4](recursos/capitulo-04/capitulo-04-regresion-lineal-simple-multiple-infografia.png)](recursos/capitulo-04/capitulo-04-regresion-lineal-simple-multiple-infografia.png)

**Video del capítulo:** <https://www.youtube.com/watch?v=SE6DCeU9h1w>

**Curso completo en YouTube:** <https://www.youtube.com/playlist?list=PLDJYd2v7Kt-Q>

[Consultar todos los videos del curso](https://www.youtube.com/playlist?list=PLDJYd2v7Kt-Q)
